# Building Performance by Net Rentable Area

This notebook generates charts showing building performance metrics (occupancy rates and rental rates) segmented by building size.

**Data Sources:**
- Office: `quarterly_report_data_office` (Supabase)
- Industrial: `quarterly_report_data_industrial` (Supabase)

**Filters:**
- `aquila_competitive_set = True`
- `building_status = 'Existing'`

**Charts Generated:**
1. Office Occupancy Rate by Building Size
2. Office Weighted Average Rent by Building Size
3. Industrial Occupancy Rate by Building Size
4. Industrial Weighted Average Rent by Building Size

In [1]:
# Imports
from dotenv import load_dotenv
from aquila_graphing_tools import initialize_supabase_connection, aquila_styled_line_chart, AQUILA_COLORS, AQUILA_FONT
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Load environment variables
load_dotenv('aquila_graph.env')

# Initialize Supabase connection
supabase = initialize_supabase_connection()
print("Connected to Supabase")

Connected to Supabase


## 1. Fetch Office Data

In [2]:
# Fetch office data with pagination to ensure more than 1000 records are loaded
print("Fetching office data...")

all_office_records = []
page = 0
page_size = 1000  # Default supabase page size

while True:
    response_office = (
        supabase.table('quarterly_report_data_office')
        .select('*')
        .eq('aquila_competitive_set', True)
        .eq('building_status', 'Existing')
        .range(page * page_size, (page + 1) * page_size - 1)
        .execute()
    )
    batch = response_office.data
    all_office_records.extend(batch)
    print(f"Fetched {len(batch):,} office records from page {page+1}")
    if len(batch) < page_size:
        break
    page += 1

df_office = pd.DataFrame(all_office_records)
print(f"Loaded {len(df_office):,} office records (all pages)")
print(f"Columns: {df_office.columns.tolist()}")
print(f"\nSample data:")
print(df_office.head())

Fetching office data...
Fetched 1,000 office records from page 1
Fetched 1,000 office records from page 2
Fetched 1,000 office records from page 3
Fetched 1,000 office records from page 4
Fetched 1,000 office records from page 5
Fetched 1,000 office records from page 6
Fetched 1,000 office records from page 7
Fetched 1,000 office records from page 8
Fetched 1,000 office records from page 9
Fetched 1,000 office records from page 10
Fetched 1,000 office records from page 11
Fetched 1,000 office records from page 12
Fetched 936 office records from page 13
Loaded 12,936 office records (all pages)
Columns: ['costar_property_id', 'property_address', 'property_name', 'city', 'building_park', 'year_built', 'year_renovated', 'tenancy', 'amenities', 'building_class', 'core_factor', 'developer_name', 'energy_star', 'aquila_micromarket', 'number_of_stories', 'leed_certified', 'property_manager_name', 'owner_name', 'parking_ratio', 'property_type', 'typical_floor_size', 'secondary_type', 'submarket

## 2. Fetch Industrial Data

In [3]:
# Fetch industrial data with pagination to ensure more than 1000 records are loaded
print("Fetching industrial data...")

all_industrial_records = []
page = 0
page_size = 1000  # Default supabase page size

while True:
    response_industrial = (
        supabase.table('quarterly_report_data_industrial')
        .select('*')
        .eq('aquila_competitive_set', True)
        .eq('building_status', 'Existing')
        .range(page * page_size, (page + 1) * page_size - 1)
        .execute()
    )
    batch = response_industrial.data
    all_industrial_records.extend(batch)
    print(f"Fetched {len(batch):,} industrial records from page {page+1}")
    if len(batch) < page_size:
        break
    page += 1

df_industrial = pd.DataFrame(all_industrial_records)
print(f"Loaded {len(df_industrial):,} industrial records (all pages)")
print(f"Columns: {df_industrial.columns.tolist()}")
print(f"\nSample data:")
print(df_industrial.head())

Fetching industrial data...
Fetched 1,000 industrial records from page 1
Fetched 1,000 industrial records from page 2
Fetched 1,000 industrial records from page 3
Fetched 1,000 industrial records from page 4
Fetched 1,000 industrial records from page 5
Fetched 1,000 industrial records from page 6
Fetched 1,000 industrial records from page 7
Fetched 1,000 industrial records from page 8
Fetched 1,000 industrial records from page 9
Fetched 1,000 industrial records from page 10
Fetched 1,000 industrial records from page 11
Fetched 1,000 industrial records from page 12
Fetched 1,000 industrial records from page 13
Fetched 1,000 industrial records from page 14
Fetched 1,000 industrial records from page 15
Fetched 1,000 industrial records from page 16
Fetched 1,000 industrial records from page 17
Fetched 1,000 industrial records from page 18
Fetched 1,000 industrial records from page 19
Fetched 1,000 industrial records from page 20
Fetched 1,000 industrial records from page 21
Fetched 1,000 i

## 3. Data Cleaning and Preparation

In [4]:
# Parse dates (handling 'quarter' format like '2025 Q4')
import re

# Clean office data
print("\n=== OFFICE DATA CLEANING ===")

# Identify the date column
date_columns = [col for col in df_office.columns if 'date' in col.lower() or 'quarter' in col.lower()]
print(f"Potential date columns: {date_columns}")

def quarter_string_to_date(q_str):
    """
    Convert 'YYYY Qn' to a datetime representing the first day of that quarter.
    Example: '2025 Q4' -> datetime(2025, 10, 1)
    """
    match = re.match(r"(\d{4})\s*[Qq](\d)", str(q_str))
    if match:
        year = int(match.group(1))
        quarter = int(match.group(2))
        month = 3 * (quarter - 1) + 1
        return pd.Timestamp(year=year, month=month, day=1)
    return pd.NaT

if 'quarter' in df_office.columns:
    df_office['date'] = df_office['quarter'].apply(quarter_string_to_date)
elif 'report_date' in df_office.columns:
    df_office['date'] = pd.to_datetime(df_office['report_date'], errors='coerce')
else:
    # Try to find any date column
    for col in date_columns:
        df_office['date'] = pd.to_datetime(df_office[col], errors='coerce')
        if df_office['date'].notna().any():
            print(f"Using '{col}' as date column")
            break

# Convert numeric columns
df_office['rentable_building_area'] = pd.to_numeric(df_office['rentable_building_area'], errors='coerce')
df_office['occupancy_pct_total'] = pd.to_numeric(df_office['occupancy_pct_total'], errors='coerce')
df_office['rental_rate'] = pd.to_numeric(df_office['rental_rate'], errors='coerce')

# Remove rows with missing critical data
df_office_clean = df_office[
    df_office['date'].notna() & 
    df_office['rentable_building_area'].notna() & 
    (df_office['rentable_building_area'] > 0)
].copy()

print(f"Office records after cleaning: {len(df_office_clean):,}")
print(f"Date range: {df_office_clean['date'].min()} to {df_office_clean['date'].max()}")
print(f"\nRentable area statistics (Office):")
print(df_office_clean['rentable_building_area'].describe())


=== OFFICE DATA CLEANING ===
Potential date columns: ['quarter']
Office records after cleaning: 12,936
Date range: 2018-01-01 00:00:00 to 2025-10-01 00:00:00

Rentable area statistics (Office):
count    1.293600e+04
mean     1.264121e+05
std      1.174999e+05
min      1.033000e+04
25%      5.582800e+04
50%      9.555500e+04
75%      1.566820e+05
max      1.179740e+06
Name: rentable_building_area, dtype: float64


In [5]:
# Clean industrial data
print("\n=== INDUSTRIAL DATA CLEANING ===")

# Identify the date column
date_columns = [col for col in df_industrial.columns if 'date' in col.lower() or 'quarter' in col.lower()]
print(f"Potential date columns: {date_columns}")

if 'quarter' in df_industrial.columns:
    df_industrial['date'] = df_industrial['quarter'].apply(quarter_string_to_date)
elif 'report_date' in df_industrial.columns:
    df_industrial['date'] = pd.to_datetime(df_industrial['report_date'], errors='coerce')
else:
    for col in date_columns:
        df_industrial['date'] = pd.to_datetime(df_industrial[col], errors='coerce')
        if df_industrial['date'].notna().any():
            print(f"Using '{col}' as date column")
            break

# Convert numeric columns
df_industrial['rentable_building_area'] = pd.to_numeric(df_industrial['rentable_building_area'], errors='coerce')
df_industrial['occupancy_pct_total'] = pd.to_numeric(df_industrial['occupancy_pct_total'], errors='coerce')
df_industrial['survey_rental_rate'] = pd.to_numeric(df_industrial['survey_rental_rate'], errors='coerce')

# Remove rows with missing critical data
df_industrial_clean = df_industrial[
    df_industrial['date'].notna() & 
    df_industrial['rentable_building_area'].notna() & 
    (df_industrial['rentable_building_area'] > 0)
].copy()

print(f"Industrial records after cleaning: {len(df_industrial_clean):,}")
print(f"Date range: {df_industrial_clean['date'].min()} to {df_industrial_clean['date'].max()}")
print(f"\nRentable area statistics (Industrial):")
print(df_industrial_clean['rentable_building_area'].describe())


=== INDUSTRIAL DATA CLEANING ===
Potential date columns: ['quarter', 'date_created', 'updated_at', 'inventory_updated_at']
Industrial records after cleaning: 26,654
Date range: 2018-04-01 00:00:00 to 2025-10-01 00:00:00

Rentable area statistics (Industrial):
count     26654.000000
mean      63403.403204
std       63535.895750
min        4942.000000
25%       22206.000000
50%       44594.000000
75%       80987.000000
max      855000.000000
Name: rentable_building_area, dtype: float64


## 4. Create Size Bins

In [6]:
# Create 5 bins for office with rounded ranges
office_min = df_office_clean['rentable_building_area'].min()
office_max = df_office_clean['rentable_building_area'].max()
office_quartiles = df_office_clean['rentable_building_area'].quantile([0.2, 0.4, 0.6, 0.8]).values

print("\n=== OFFICE SIZE BINS ===")
print(f"Min: {office_min:,.0f} SF")
print(f"20th percentile: {office_quartiles[0]:,.0f} SF")
print(f"40th percentile: {office_quartiles[1]:,.0f} SF")
print(f"60th percentile: {office_quartiles[2]:,.0f} SF")
print(f"80th percentile: {office_quartiles[3]:,.0f} SF")
print(f"Max: {office_max:,.0f} SF")

# Round to create readable bins
def round_to_readable(value):
    """Round to nearest 5k, 10k, 25k, 50k, or 100k depending on magnitude"""
    if value < 10000:
        return round(value / 5000) * 5000
    elif value < 50000:
        return round(value / 10000) * 10000
    elif value < 100000:
        return round(value / 25000) * 25000
    else:
        return round(value / 50000) * 50000

office_bins = [
    0,
    round_to_readable(office_quartiles[0]),
    round_to_readable(office_quartiles[1]),
    round_to_readable(office_quartiles[2]),
    round_to_readable(office_quartiles[3]),
    float('inf')
]

office_labels = [
    f"0-{office_bins[1]/1000:.0f}k SF",
    f"{office_bins[1]/1000:.0f}k-{office_bins[2]/1000:.0f}k SF",
    f"{office_bins[2]/1000:.0f}k-{office_bins[3]/1000:.0f}k SF",
    f"{office_bins[3]/1000:.0f}k-{office_bins[4]/1000:.0f}k SF",
    f"{office_bins[4]/1000:.0f}k+ SF"
]

print(f"\nOffice bins: {office_bins}")
print(f"Office labels: {office_labels}")

df_office_clean['size_bin'] = pd.cut(
    df_office_clean['rentable_building_area'],
    bins=office_bins,
    labels=office_labels,
    include_lowest=True
)


=== OFFICE SIZE BINS ===
Min: 10,330 SF
20th percentile: 48,798 SF
40th percentile: 79,384 SF
60th percentile: 115,206 SF
80th percentile: 178,606 SF
Max: 1,179,740 SF

Office bins: [0, 50000, 75000, 100000, 200000, inf]
Office labels: ['0-50k SF', '50k-75k SF', '75k-100k SF', '100k-200k SF', '200k+ SF']


In [7]:
# Create 5 bins for industrial with rounded ranges
industrial_min = df_industrial_clean['rentable_building_area'].min()
industrial_max = df_industrial_clean['rentable_building_area'].max()
industrial_quartiles = df_industrial_clean['rentable_building_area'].quantile([0.2, 0.4, 0.6, 0.8]).values

print("\n=== INDUSTRIAL SIZE BINS ===")
print(f"Min: {industrial_min:,.0f} SF")
print(f"20th percentile: {industrial_quartiles[0]:,.0f} SF")
print(f"40th percentile: {industrial_quartiles[1]:,.0f} SF")
print(f"60th percentile: {industrial_quartiles[2]:,.0f} SF")
print(f"80th percentile: {industrial_quartiles[3]:,.0f} SF")
print(f"Max: {industrial_max:,.0f} SF")

industrial_bins = [
    0,
    round_to_readable(industrial_quartiles[0]),
    round_to_readable(industrial_quartiles[1]),
    round_to_readable(industrial_quartiles[2]),
    round_to_readable(industrial_quartiles[3]),
    float('inf')
]

industrial_labels = [
    f"0-{industrial_bins[1]/1000:.0f}k SF",
    f"{industrial_bins[1]/1000:.0f}k-{industrial_bins[2]/1000:.0f}k SF",
    f"{industrial_bins[2]/1000:.0f}k-{industrial_bins[3]/1000:.0f}k SF",
    f"{industrial_bins[3]/1000:.0f}k-{industrial_bins[4]/1000:.0f}k SF",
    f"{industrial_bins[4]/1000:.0f}k+ SF"
]

print(f"\nIndustrial bins: {industrial_bins}")
print(f"Industrial labels: {industrial_labels}")

df_industrial_clean['size_bin'] = pd.cut(
    df_industrial_clean['rentable_building_area'],
    bins=industrial_bins,
    labels=industrial_labels,
    include_lowest=True
)


=== INDUSTRIAL SIZE BINS ===
Min: 4,942 SF
20th percentile: 18,730 SF
40th percentile: 34,650 SF
60th percentile: 56,100 SF
80th percentile: 95,000 SF
Max: 855,000 SF

Industrial bins: [0, 20000, 30000, 50000, 100000, inf]
Industrial labels: ['0-20k SF', '20k-30k SF', '30k-50k SF', '50k-100k SF', '100k+ SF']


## 5. Calculate Weighted Metrics by Date and Size Bin

In [8]:
# Office - Weighted occupancy rate
print("\n=== CALCULATING OFFICE WEIGHTED OCCUPANCY ===")

office_occ_by_size = df_office_clean.groupby(['date', 'size_bin']).apply(
    lambda x: np.average(
        x['occupancy_pct_total'].dropna(),
        weights=x.loc[x['occupancy_pct_total'].notna(), 'rentable_building_area']
    ) if len(x['occupancy_pct_total'].dropna()) > 0 else np.nan
).reset_index(name='weighted_occupancy_pct')

print(f"Office occupancy data shape: {office_occ_by_size.shape}")
print(office_occ_by_size.head())


=== CALCULATING OFFICE WEIGHTED OCCUPANCY ===
Office occupancy data shape: (160, 3)
        date      size_bin  weighted_occupancy_pct
0 2018-01-01      0-50k SF                0.897027
1 2018-01-01    50k-75k SF                0.876040
2 2018-01-01   75k-100k SF                0.897558
3 2018-01-01  100k-200k SF                0.900151
4 2018-01-01      200k+ SF                0.910487


C:\Users\NLin\AppData\Local\Temp\ipykernel_24704\104213432.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  office_occ_by_size = df_office_clean.groupby(['date', 'size_bin']).apply(
C:\Users\NLin\AppData\Local\Temp\ipykernel_24704\104213432.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  office_occ_by_size = df_office_clean.groupby(['date', 'size_bin']).apply(


In [9]:
# Office - Weighted average rent
print("\n=== CALCULATING OFFICE WEIGHTED RENT ===")

office_rent_by_size = df_office_clean.groupby(['date', 'size_bin']).apply(
    lambda x: np.average(
        x['rental_rate'].dropna(),
        weights=x.loc[x['rental_rate'].notna(), 'rentable_building_area']
    ) if len(x['rental_rate'].dropna()) > 0 else np.nan
).reset_index(name='weighted_avg_rent')

print(f"Office rent data shape: {office_rent_by_size.shape}")
print(office_rent_by_size.head())


=== CALCULATING OFFICE WEIGHTED RENT ===
Office rent data shape: (160, 3)
        date      size_bin  weighted_avg_rent
0 2018-01-01      0-50k SF          20.775595
1 2018-01-01    50k-75k SF          22.744597
2 2018-01-01   75k-100k SF          24.419687
3 2018-01-01  100k-200k SF          27.010259
4 2018-01-01      200k+ SF          32.953655


C:\Users\NLin\AppData\Local\Temp\ipykernel_24704\4230076423.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  office_rent_by_size = df_office_clean.groupby(['date', 'size_bin']).apply(
C:\Users\NLin\AppData\Local\Temp\ipykernel_24704\4230076423.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  office_rent_by_size = df_office_clean.groupby(['date', 'size_bin']).apply(


In [10]:
# Industrial - Weighted occupancy rate
print("\n=== CALCULATING INDUSTRIAL WEIGHTED OCCUPANCY ===")

industrial_occ_by_size = df_industrial_clean.groupby(['date', 'size_bin']).apply(
    lambda x: np.average(
        x['occupancy_pct_total'].dropna(),
        weights=x.loc[x['occupancy_pct_total'].notna(), 'rentable_building_area']
    ) if len(x['occupancy_pct_total'].dropna()) > 0 else np.nan
).reset_index(name='weighted_occupancy_pct')

print(f"Industrial occupancy data shape: {industrial_occ_by_size.shape}")
print(industrial_occ_by_size.head())


=== CALCULATING INDUSTRIAL WEIGHTED OCCUPANCY ===


C:\Users\NLin\AppData\Local\Temp\ipykernel_24704\1679506866.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  industrial_occ_by_size = df_industrial_clean.groupby(['date', 'size_bin']).apply(


Industrial occupancy data shape: (155, 3)
        date     size_bin  weighted_occupancy_pct
0 2018-04-01     0-20k SF                0.931261
1 2018-04-01   20k-30k SF                0.921506
2 2018-04-01   30k-50k SF                0.923743
3 2018-04-01  50k-100k SF                0.906873
4 2018-04-01     100k+ SF                0.911359


C:\Users\NLin\AppData\Local\Temp\ipykernel_24704\1679506866.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  industrial_occ_by_size = df_industrial_clean.groupby(['date', 'size_bin']).apply(


In [11]:
# Industrial - Weighted average rent
print("\n=== CALCULATING INDUSTRIAL WEIGHTED RENT ===")

industrial_rent_by_size = df_industrial_clean.groupby(['date', 'size_bin']).apply(
    lambda x: np.average(
        x['survey_rental_rate'].dropna(),
        weights=x.loc[x['survey_rental_rate'].notna(), 'rentable_building_area']
    ) if len(x['survey_rental_rate'].dropna()) > 0 else np.nan
).reset_index(name='weighted_avg_rent')

print(f"Industrial rent data shape: {industrial_rent_by_size.shape}")
print(industrial_rent_by_size.tail())

C:\Users\NLin\AppData\Local\Temp\ipykernel_24704\887294186.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  industrial_rent_by_size = df_industrial_clean.groupby(['date', 'size_bin']).apply(



=== CALCULATING INDUSTRIAL WEIGHTED RENT ===
Industrial rent data shape: (155, 3)
          date     size_bin  weighted_avg_rent
150 2025-10-01     0-20k SF          17.053872
151 2025-10-01   20k-30k SF          15.622974
152 2025-10-01   30k-50k SF          16.523477
153 2025-10-01  50k-100k SF          14.722372
154 2025-10-01     100k+ SF          11.993777


C:\Users\NLin\AppData\Local\Temp\ipykernel_24704\887294186.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  industrial_rent_by_size = df_industrial_clean.groupby(['date', 'size_bin']).apply(


## 6. Generate Charts

In [12]:
# Chart 1: Office Occupancy Rate by Building Size
fig_office_occ = aquila_styled_line_chart(
    office_occ_by_size,
    x='date',
    y='weighted_occupancy_pct',
    color='size_bin',
    title='Office Occupancy Rate by Building Size (Weighted by Rentable Area)'
)
fig_office_occ.update_yaxes(tickformat='.1%', title='Occupancy Rate')
fig_office_occ.update_xaxes(title='Quarter')
fig_office_occ.write_html('charts/office/office_occupancy_by_size.html')
print("Saved: charts/office_occupancy_by_size.html")
fig_office_occ.show()

Saved: charts/office_occupancy_by_size.html


In [13]:
# Chart 2: Office Weighted Average Rent by Building Size
fig_office_rent = aquila_styled_line_chart(
    office_rent_by_size,
    x='date',
    y='weighted_avg_rent',
    color='size_bin',
    title='Office Weighted Average Rent by Building Size'
)
fig_office_rent.update_yaxes(tickprefix='$', tickformat=',.2f', title='Rent ($/SF)')
fig_office_rent.update_xaxes(title='Quarter')
fig_office_rent.write_html('charts/office/office_rent_by_size.html')
print("Saved: charts/office_rent_by_size.html")
fig_office_rent.show()

Saved: charts/office_rent_by_size.html


In [14]:
# Chart 3: Industrial Occupancy Rate by Building Size
fig_industrial_occ = aquila_styled_line_chart(
    industrial_occ_by_size,
    x='date',
    y='weighted_occupancy_pct',
    color='size_bin',
    title='Industrial Occupancy Rate by Building Size (Weighted by Rentable Area)'
)
fig_industrial_occ.update_yaxes(tickformat='.1%', title='Occupancy Rate')
fig_industrial_occ.update_xaxes(title='Quarter')
fig_industrial_occ.write_html('charts/industrial/industrial_occupancy_by_size.html')
print("Saved: charts/industrial_occupancy_by_size.html")
fig_industrial_occ.show()

Saved: charts/industrial_occupancy_by_size.html


In [15]:
# Chart 4: Industrial Weighted Average Rent by Building Size
# Drop missing quarters (rows with missing 'date' or 'weighted_avg_rent')
industrial_rent_by_size_clean = industrial_rent_by_size.dropna(subset=['date', 'weighted_avg_rent'])

fig_industrial_rent = aquila_styled_line_chart(
    industrial_rent_by_size_clean,
    x='date',
    y='weighted_avg_rent',
    color='size_bin',
    title='Industrial Weighted Average Rent by Building Size'
)
fig_industrial_rent.update_yaxes(tickprefix='$', tickformat=',.2f', title='Rent ($/SF)')
fig_industrial_rent.update_xaxes(title='Quarter')
fig_industrial_rent.write_html('charts/industrial/industrial_rent_by_size.html')
print("Saved: charts/industrial_rent_by_size.html")
fig_industrial_rent.show()

Saved: charts/industrial_rent_by_size.html


## 7. Summary Statistics

In [16]:
print("\n=== SUMMARY ===")
print(f"\nOffice:")
print(f"  - Total buildings tracked: {df_office_clean['building_id'].nunique() if 'building_id' in df_office_clean.columns else 'N/A'}")
print(f"  - Date range: {office_occ_by_size['date'].min().strftime('%Y-%m-%d')} to {office_occ_by_size['date'].max().strftime('%Y-%m-%d')}")
print(f"  - Size bins: {office_labels}")

print(f"\nIndustrial:")
print(f"  - Total buildings tracked: {df_industrial_clean['building_id'].nunique() if 'building_id' in df_industrial_clean.columns else 'N/A'}")
print(f"  - Date range: {industrial_occ_by_size['date'].min().strftime('%Y-%m-%d')} to {industrial_occ_by_size['date'].max().strftime('%Y-%m-%d')}")
print(f"  - Size bins: {industrial_labels}")

print("\nAll charts saved to charts/ directory!")


=== SUMMARY ===

Office:
  - Total buildings tracked: N/A
  - Date range: 2018-01-01 to 2025-10-01
  - Size bins: ['0-50k SF', '50k-75k SF', '75k-100k SF', '100k-200k SF', '200k+ SF']

Industrial:
  - Total buildings tracked: N/A
  - Date range: 2018-04-01 to 2025-10-01
  - Size bins: ['0-20k SF', '20k-30k SF', '30k-50k SF', '50k-100k SF', '100k+ SF']

All charts saved to charts/ directory!
